# 03 — Comparar o comparável

**A pergunta:** em mercados com menos companhias, a tarifa por quilômetro
é mais alta, **comparando distâncias parecidas**?

O notebook 02 comparou os mercados sem controle nenhum e não chegou a uma
resposta: os mercados com um concorrente só são mais longos, e a
distância deixa o quilômetro mais barato e a passagem mais cara. Aqui a
comparação é refeita entre mercados de distância parecida, primeiro
dentro de faixas e depois numa regressão.

Faz parte da [etapa 7](../README.md#etapas) do projeto. Os dados vêm de
`dados/marts/mercado_mes.parquet` (um registro por mercado, mês e grupo) e
de `dados/marts/dim_mercado.parquet` (a distância de cada mercado), os dois
montados pelo `construir.py` a partir da coleta de 13/10/2026. As
definições são as do notebook 02: concorrente é o grupo com pelo menos 5%
dos passageiros de voo regular do mercado no mês, e a tarifa por km é a
receita dividida pelos assentos-km, as duas somadas.

## 1. As faixas de distância, declaradas antes de olhar

Cada mercado entra numa de quatro faixas, pela distância em linha reta
entre as duas cidades:

| Faixa | Distância |
|---|---|
| curta | até 500 km |
| média | de 500 a 1.000 km |
| longa | de 1.000 a 2.000 km |
| muito longa | acima de 2.000 km |

As fronteiras são números redondos, escolhidas antes de qualquer conta por
faixa, e não mudam depois de ver o resultado: fronteira escolhida olhando
a tarifa acharia a diferença que se quisesse. Dentro de cada faixa, a
comparação é a do notebook 02, ano a ano, pelo ano da venda.

In [1]:
import pandas as pd

URL = "https://raw.githubusercontent.com/ivadivad/voos-em-numeros/main/dados/marts/mercado_mes.parquet"
DIM_MERCADO = "https://raw.githubusercontent.com/ivadivad/voos-em-numeros/main/dados/marts/dim_mercado.parquet"
mart = pd.read_parquet(URL)
distancia = pd.read_parquet(DIM_MERCADO).set_index("mercado")["distancia_km"].astype(float)

# um registro por mercado e mes: as somas dos grupos e a concentracao do mercado
mercados = mart.groupby(["mes", "mercado"]).agg(
    concorrentes=("concorrentes", "first"), passageiros=("passageiros_pagos", "sum"),
    receita=("receita", "sum"), assentos=("assentos_vendidos", "sum")).reset_index()
mercados["mes"] = pd.to_datetime(mercados["mes"])
mercados["ano"] = mercados["mes"].dt.year
mercados["distancia_km"] = mercados["mercado"].map(distancia)
mercados["assentos_km"] = mercados["assentos"] * mercados["distancia_km"]
CLASSES = ["1", "2", "3 ou mais"]
mercados["classe"] = mercados["concorrentes"].clip(upper=3).map(dict(zip([1, 2, 3], CLASSES)))
FAIXAS = ["até 500 km", "500 a 1.000 km", "1.000 a 2.000 km", "acima de 2.000 km"]
mercados["faixa"] = pd.cut(mercados["distancia_km"], [0, 500, 1000, 2000, float("inf")], labels=FAIXAS)

com_tarifa = mercados[(mercados["assentos"] > 0) & mercados["distancia_km"].notna()].copy()
print(f"{len(mercados)} mercado-meses; {len(com_tarifa)} com tarifa vendida e distância, "
      f"{mercados['assentos'].eq(0).sum()} sem tarifa e {mercados['distancia_km'].isna().sum()} sem distância")

30886 mercado-meses; 29554 com tarifa vendida e distância, 1329 sem tarifa e 3 sem distância


A unidade daqui em diante é o mercado no mês: as somas de todos os grupos
e o número de concorrentes do mercado. Entram os 29.554 mercado-meses com
tarifa vendida e distância conhecida. Ficam de fora 1.329 sem nenhuma
tarifa vendida no mês e 3 sem distância.

## 2. A tarifa por km dentro de cada faixa

Primeiro a composição: quantos mercados de cada tipo há em cada faixa, e
se a faixa aproxima a distância dos mercados de um concorrente da dos
mercados de três ou mais.

In [2]:
ano = com_tarifa[com_tarifa["ano"] == 2025]
s = ano.groupby(["faixa", "classe"], observed=True).agg(
    mercado_meses=("mercado", "size"), assentos=("assentos", "sum"), assentos_km=("assentos_km", "sum"))
s["% dos assentos da faixa"] = s["assentos"] / s.groupby(level="faixa", observed=True)["assentos"].transform("sum") * 100
s["distância média (km)"] = s["assentos_km"] / s["assentos"]
print("2025, por faixa e número de concorrentes")
print(s[["mercado_meses", "% dos assentos da faixa", "distância média (km)"]].round(0).to_string())

2025, por faixa e número de concorrentes
                             mercado_meses  % dos assentos da faixa  distância média (km)
faixa             classe                                                                 
até 500 km        1                   1022                     12.0                 385.0
                  2                    117                     16.0                 391.0
                  3 ou mais             99                     73.0                 371.0
500 a 1.000 km    1                    639                     12.0                 780.0
                  2                    209                     22.0                 763.0
                  3 ou mais            191                     66.0                 723.0
1.000 a 2.000 km  1                    715                     21.0                1519.0
                  2                    324                     31.0                1390.0
                  3 ou mais            159                 

**Dentro das faixas, a distância média quase se iguala.** Em 2025, até
500 km, os mercados de um concorrente têm 385 km em média e os de três ou
mais, 371; de 1.000 a 2.000 km, 1.519 e 1.518. A distância que sobra é
maior de 500 a 1.000 km (780 e 723) e acima de 2.000 km (2.459 e 2.270),
sempre com os de um concorrente mais longos. Em toda faixa, eles são os
mais numerosos e levam de 12% a 21% dos assentos vendidos.

In [3]:
def tarifa_km(registros, por):
    """Tarifa por km da razao entre as somas, em cada combinacao de `por`."""
    s = registros.groupby(por, observed=True)[["receita", "assentos_km"]].sum()
    return s["receita"] / s["assentos_km"]


print("tarifa por km em 2025, todas as classes juntas (R$)")
print(tarifa_km(com_tarifa[com_tarifa["ano"] == 2025], "faixa").round(3).to_string(), "\n")

por_faixa = tarifa_km(com_tarifa, ["faixa", "ano", "classe"]).unstack("classe")
print("tarifa por km em 2025 (R$)")
print(por_faixa.xs(2025, level="ano").round(3).to_string(), "\n")

completos = por_faixa.dropna()
resumo = pd.DataFrame({
    "anos com as três classes": completos.groupby(level="faixa", observed=True).size(),
    "1 acima de 3 ou mais": (completos["1"] > completos["3 ou mais"]).groupby(level="faixa", observed=True).sum(),
    "2 entre 1 e 3 ou mais": ((completos["2"] < completos["1"]) & (completos["2"] > completos["3 ou mais"]))
                             .groupby(level="faixa", observed=True).sum(),
    "1 sobre 3 ou mais, mediana dos anos (%)": ((completos["1"] / completos["3 ou mais"] - 1) * 100)
                                               .groupby(level="faixa", observed=True).median().round(1),
})
print("de 2019 a 2026, faixa a faixa")
print(resumo.to_string())

tarifa por km em 2025, todas as classes juntas (R$)
faixa
até 500 km           1.201
500 a 1.000 km       0.646
1.000 a 2.000 km     0.462
acima de 2.000 km    0.355 

tarifa por km em 2025 (R$)
classe                 1      2  3 ou mais
faixa                                     
até 500 km         1.653  0.980      1.176
500 a 1.000 km     0.817  0.670      0.603
1.000 a 2.000 km   0.474  0.511      0.429
acima de 2.000 km  0.354  0.376      0.351 

de 2019 a 2026, faixa a faixa
                   anos com as três classes  1 acima de 3 ou mais  2 entre 1 e 3 ou mais  1 sobre 3 ou mais, mediana dos anos (%)
faixa                                                                                                                            
até 500 km                                8                     8                      2                                     25.5
500 a 1.000 km                            8                     8                      7                                     

**Comparando distâncias parecidas, um concorrente só cobra mais por km em
todas as faixas até 2.000 km, todos os anos.** De 2019 a 2026, a tarifa
por km dos mercados de um concorrente fica acima da dos de três ou mais
em 8 dos 8 anos até 500 km, de 500 a 1.000 km e de 1.000 a 2.000 km, e a
diferença mediana entre os anos é de 25,5%, 23,2% e 10,5%. Acima de
2.000 km, ela some: 5 dos 8 anos, mediana de 0,7%. Em 2025, até 500 km, o
km sai R\$ 1,653 onde há um concorrente e R\$ 1,176 onde há três ou mais.

O quilômetro fica mais barato a cada faixa (em 2025, de R\$ 1,201 até
500 km a R\$ 0,355 acima de 2.000 km, todas as classes juntas), e os
mercados de um concorrente são um pouco mais longos dentro da faixa. A
distância que sobra joga contra o resultado: sem ela, a diferença seria
maior, não menor.

Os mercados de dois concorrentes seguem sem lugar fixo: ficam entre os
outros dois em 7 dos 8 anos de 500 a 1.000 km, em 5 de 1.000 a 2.000 km,
em 2 até 500 km e em 1 acima de 2.000 km.